# PlumeTrace PINN Source Attribution - Kaggle Dual T4 Notebook (v2)

Use this notebook on Kaggle with **Accelerator: GPU T4 x2**. It trains the PlumeTrace Physics-Informed Neural Network (PINN) for inverse pollutant-source attribution, saves a model checkpoint, writes a dense source-probability map, and validates whether the predicted source is close to the injected synthetic factory source.

The notebook keeps the original 2D advection-diffusion physics loss:

`dC/dt + u*dC/dx + v*dC/dy - D*(d2C/dx2 + d2C/dy2) = 0`

For numerical stability, training runs in float32 instead of mixed precision, because second-order PINN derivatives can become noisy in fp16 - this was true in v1 and remains true here; adding `torch.cuda.amp` would trade away exactly the stability this architecture needs.

### What's new in this refactor
- **Architecture factory** (`ModelConfig.arch_type`): `fourier_residual` (default - Fourier features + residual MLP + learnable-steepness Swish), `residual`, `siren`, or `mlp` (original baseline), switchable with one config field.
- **SoftAdapt adaptive loss balancing**: data/physics/boundary weights are re-derived every epoch from their relative rate of improvement instead of being hand-tuned once.
- **Residual-based Adaptive Refinement (RAR)**: periodically resamples collocation points toward the highest-PDE-residual regions of the domain.
- **Sobol quasi-random sampling** for collocation/boundary points (lower discrepancy than uniform random for the same point budget).
- **Two-stage optimization**: AdamW + warmup/cosine schedule, then a full-batch L-BFGS fine-tune pass.
- **EMA of weights**: the checkpointed/evaluated model is an exponential moving average, not the raw last-step weights.
- **Local MLflow tracking** (SQLite-backed, no account needed) alongside the existing CSV/JSON/plot outputs.
- New training-diagnostics figure (loss components, adaptive weights, LR schedule, gradient norm) next to the original probability-map figure.

### Explicitly out of scope for this pass
Distributed Optuna sweeps, W&B, MLflow-remote, Bayesian PINN / deep ensembles / K-fold cross-validation, a full ablation matrix across every architecture pair, CUDA graphs, and `torch.compile` were left out on purpose - they either need infrastructure a single Kaggle session doesn't have (accounts, multi-run orchestration) or would multiply the runtime far beyond what "keep it fully runnable end-to-end" allows. The architecture factory and config make it straightforward to add an outer sweep loop later if you want it.

In [ ]:
import json
import math
import os
import random
import subprocess
import sys
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from torch.quasirandom import SobolEngine

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd() / 'kaggle_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int = 2026) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, 'set_float32_matmul_precision'):
        torch.set_float32_matmul_precision('high')


seed_everything(2026)

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
GPU_COUNT = torch.cuda.device_count()
print('Torch:', torch.__version__)
print('Device:', DEVICE)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', GPU_COUNT)
for i in range(GPU_COUNT):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}')
if GPU_COUNT < 2:
    print('Warning: Kaggle is not currently exposing two GPUs. Enable Accelerator: GPU T4 x2 for the intended run.')

# --- Local MLflow tracking (no account/API key needed). Falls back to a no-op
# logger if mlflow is unavailable (e.g. no internet on the Kaggle session), so
# the notebook always stays runnable end-to-end. ---
try:
    import mlflow
except ImportError:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mlflow'], check=True, timeout=120)
        import mlflow
    except Exception as exc:  # noqa: BLE001 - defensive fallback, not a training-critical path
        mlflow = None
        print(f'mlflow unavailable ({exc}); continuing without experiment tracking.')

if mlflow is not None:
    mlflow.set_tracking_uri(f"sqlite:///{OUTPUT_DIR / 'mlflow.db'}")
    mlflow.set_experiment('plumetrace_pinn')
    print('MLflow tracking DB:', OUTPUT_DIR / 'mlflow.db')


def mlflow_start_run(run_name: str) -> None:
    if mlflow is not None:
        mlflow.start_run(run_name=run_name)


def mlflow_end_run() -> None:
    if mlflow is not None:
        mlflow.end_run()


def mlflow_log_params(params: dict[str, Any]) -> None:
    if mlflow is not None:
        # mlflow rejects non-primitive / overly long values; stringify defensively.
        safe = {k: (v if isinstance(v, (int, float, str, bool)) else str(v)) for k, v in params.items()}
        mlflow.log_params(safe)


def mlflow_log_metrics(metrics: dict[str, float], step: int) -> None:
    if mlflow is not None:
        mlflow.log_metrics(metrics, step=step)


def mlflow_log_artifact(path: Path) -> None:
    if mlflow is not None and path.exists():
        mlflow.log_artifact(str(path))


## Training Configuration

`ModelConfig` controls the network architecture (default: `fourier_residual`). `TrainingConfig` controls everything about the optimization: the AdamW stage (`epochs`, `learning_rate`, `warmup_epochs`), the L-BFGS fine-tune stage (`lbfgs_steps`), RAR sampling (`rar_*`), SoftAdapt (`softadapt_*`), and EMA (`ema_decay`).

`COLLOCATION_POINTS` defaults to at least 2,000 as required by the hackathon spec. On two T4 GPUs, this notebook uses 8,192 collocation points per epoch to make better use of the runtime. Set it back to `2000` if you want an exact match to the local demo script.

In [ ]:
@dataclass(frozen=True)
class SensorStation:
    sensor_id: str
    latitude: float
    longitude: float


@dataclass(frozen=True)
class CitySector:
    lat_min: float = 40.7040
    lat_max: float = 40.7220
    lon_min: float = -74.0160
    lon_max: float = -73.9940
    source_latitude: float = 40.7138
    source_longitude: float = -74.0072


@dataclass(frozen=True)
class ModelConfig:
    """Architecture factory switch. 'fourier_residual' (Fourier features +
    residual MLP + learnable-steepness Swish) is the strongest default for
    this problem; 'mlp' reproduces the original baseline for comparison,
    'residual' and 'siren' are also selectable without code changes."""

    arch_type: Literal['mlp', 'residual', 'fourier_residual', 'siren'] = 'fourier_residual'
    hidden_units: int = 128
    hidden_layers: int = 8
    fourier_bands: int = 8
    fourier_sigma: float = 4.0
    siren_omega0_first: float = 30.0
    siren_omega0_hidden: float = 30.0
    use_adaptive_activation: bool = True


@dataclass(frozen=True)
class TrainingConfig:
    epochs: int = 2500
    learning_rate: float = 1.0e-3
    weight_decay: float = 1.0e-5
    lambda_data: float = 1.0
    lambda_physics: float = 0.10
    lambda_boundary: float = 0.02
    collocation_points: int = 8192 if GPU_COUNT >= 2 else 2000
    boundary_points: int = 1024
    log_every: int = 50
    wind_u: float = 0.16
    wind_v: float = -0.06
    diffusion: float = 0.006
    sensor_time_samples: int = 96
    random_seed: int = 2026
    gradient_clip_norm: float = 5.0
    grid_size: int = 220

    # --- LR schedule (AdamW stage) ---
    warmup_epochs: int = 100
    min_lr_ratio: float = 0.08  # matches the original CosineAnnealingLR eta_min ratio

    # --- Exponential Moving Average of weights, evaluated/checkpointed instead
    # of the raw final-step weights for a less noisy, better-generalizing model ---
    ema_decay: float = 0.999

    # --- Residual-based Adaptive Refinement (RAR) collocation sampling ---
    rar_interval: int = 100          # refresh high-residual points every N epochs
    rar_fraction: float = 0.25       # fraction of collocation_points drawn from RAR
    rar_pool_multiplier: int = 4     # candidate pool size = collocation_points * this
    rar_warmup_epochs: int = 200     # let the field settle before trusting residuals

    # --- SoftAdapt adaptive loss balancing ---
    softadapt_beta: float = 0.1
    softadapt_floor: float = 0.05
    softadapt_ceil: float = 5.0
    softadapt_warmup_epochs: int = 20

    # --- Stage-2 L-BFGS fine-tuning (full-batch second-order refinement after
    # the AdamW stage converges to a good basin) ---
    lbfgs_steps: int = 200
    lbfgs_lr: float = 0.5


SENSOR_STATIONS = (
    SensorStation('industrial_north', 40.7180, -74.0060),
    SensorStation('residential_east', 40.7140, -73.9980),
    SensorStation('park_south', 40.7080, -74.0040),
    SensorStation('river_west', 40.7120, -74.0120),
)

SECTOR = CitySector()
MODEL_CONFIG = ModelConfig()
CONFIG = TrainingConfig()
print(MODEL_CONFIG)
print(CONFIG)


In [ ]:
def lat_lon_to_normalized(latitude: float | np.ndarray, longitude: float | np.ndarray, sector: CitySector) -> tuple[np.ndarray, np.ndarray]:
    lon_arr = np.asarray(longitude, dtype=np.float32)
    lat_arr = np.asarray(latitude, dtype=np.float32)
    x = (lon_arr - sector.lon_min) / (sector.lon_max - sector.lon_min)
    y = (lat_arr - sector.lat_min) / (sector.lat_max - sector.lat_min)
    return x.astype(np.float32), y.astype(np.float32)

def normalized_to_lat_lon(x: np.ndarray, y: np.ndarray, sector: CitySector) -> tuple[np.ndarray, np.ndarray]:
    longitude = sector.lon_min + x * (sector.lon_max - sector.lon_min)
    latitude = sector.lat_min + y * (sector.lat_max - sector.lat_min)
    return latitude.astype(np.float32), longitude.astype(np.float32)

def haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    radius = 6_371_000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2.0) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2.0) ** 2
    return 2.0 * radius * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))

def analytic_advection_diffusion_plume(
    x: np.ndarray,
    y: np.ndarray,
    t: np.ndarray,
    source_x: float | np.ndarray,
    source_y: float | np.ndarray,
    wind_u: float,
    wind_v: float,
    diffusion: float,
    source_strength: float = 1.0,
    initial_spread: float = 0.035,
) -> np.ndarray:
    effective_time = np.maximum(t, 0.0) + initial_spread
    advected_x = source_x + wind_u * t
    advected_y = source_y + wind_v * t
    radial_distance_squared = (x - advected_x) ** 2 + (y - advected_y) ** 2
    denominator = 4.0 * math.pi * diffusion * effective_time
    exponent = -radial_distance_squared / (4.0 * diffusion * effective_time)
    return source_strength * np.exp(exponent) / denominator

@dataclass
class SyntheticSensorDataset:
    features: Tensor
    concentration: Tensor
    rows: list[dict[str, Any]]
    concentration_scale_ppb: float
    source_x: float
    source_y: float

def generate_synthetic_sensor_data(sector: CitySector, config: TrainingConfig, device: torch.device) -> SyntheticSensorDataset:
    source_x_arr, source_y_arr = lat_lon_to_normalized(sector.source_latitude, sector.source_longitude, sector)
    source_x = float(np.asarray(source_x_arr))
    source_y = float(np.asarray(source_y_arr))
    rows: list[dict[str, Any]] = []
    x_values: list[float] = []
    y_values: list[float] = []
    t_values: list[float] = []
    plume_values: list[float] = []
    sample_times = np.linspace(0.05, 1.0, config.sensor_time_samples, dtype=np.float32)
    for sensor in SENSOR_STATIONS:
        sensor_x, sensor_y = lat_lon_to_normalized(sensor.latitude, sensor.longitude, sector)
        sx = float(np.asarray(sensor_x))
        sy = float(np.asarray(sensor_y))
        for elapsed_time in sample_times:
            signal = analytic_advection_diffusion_plume(
                np.asarray(sx, dtype=np.float32),
                np.asarray(sy, dtype=np.float32),
                np.asarray(float(elapsed_time), dtype=np.float32),
                source_x,
                source_y,
                config.wind_u,
                config.wind_v,
                config.diffusion,
            )
            x_values.append(sx)
            y_values.append(sy)
            t_values.append(float(elapsed_time))
            plume_values.append(float(signal))
    plume = np.asarray(plume_values, dtype=np.float32)
    normalized_signal = plume / max(float(plume.max()), 1.0e-8)
    background_so2_ppb = 7.5
    event_scale_ppb = 180.0
    noise = np.random.normal(loc=0.0, scale=1.65, size=normalized_signal.shape)
    so2_ppb = np.clip(background_so2_ppb + event_scale_ppb * normalized_signal + noise, 0.0, None)
    concentration_scale_ppb = float(np.max(so2_ppb))
    target = (so2_ppb / concentration_scale_ppb).astype(np.float32)
    for i, (x, y, t, so2, c) in enumerate(zip(x_values, y_values, t_values, so2_ppb, target)):
        sensor = SENSOR_STATIONS[i // config.sensor_time_samples]
        rows.append({
            'sensor_id': sensor.sensor_id,
            'latitude': sensor.latitude,
            'longitude': sensor.longitude,
            'x': float(x),
            'y': float(y),
            'elapsed_time': float(t),
            'so2_ppb': float(so2),
            'normalized_concentration': float(c),
        })
    features = torch.tensor(np.column_stack([x_values, y_values, t_values]), dtype=torch.float32, device=device)
    concentration = torch.tensor(target, dtype=torch.float32, device=device).view(-1, 1)
    print(f'Generated {len(rows)} synthetic sensor readings from {len(SENSOR_STATIONS)} sensors')
    print(f'Known source lat/lon: ({sector.source_latitude:.6f}, {sector.source_longitude:.6f})')
    return SyntheticSensorDataset(features, concentration, rows, concentration_scale_ppb, source_x, source_y)

dataset = generate_synthetic_sensor_data(SECTOR, CONFIG, DEVICE)
pd.DataFrame(dataset.rows).head()


In [ ]:
# ============================================================
# Architecture factory
# ============================================================
class FourierFeatures(nn.Module):
    """Random Fourier feature positional encoding (Tancik et al., 2020).
    Lets the network represent the sharp, multi-scale spatial gradients of a
    plume field far more easily than raw (x, y, t) coordinates can."""

    def __init__(self, in_dim: int, num_bands: int, sigma: float, seed: int = 0) -> None:
        super().__init__()
        generator = torch.Generator().manual_seed(seed)
        b_matrix = torch.randn((in_dim, num_bands), generator=generator) * sigma
        self.register_buffer('b_matrix', b_matrix)

    def forward(self, x: Tensor) -> Tensor:
        projected = 2.0 * math.pi * (x @ self.b_matrix)
        return torch.cat([torch.sin(projected), torch.cos(projected)], dim=-1)


class AdaptiveSwish(nn.Module):
    """Swish/SiLU with a learnable per-channel steepness beta (an 'adaptive
    activation' in the PINN literature; speeds convergence vs. fixed Tanh)."""

    def __init__(self, num_features: int) -> None:
        super().__init__()
        self.beta = nn.Parameter(torch.ones(num_features))

    def forward(self, x: Tensor) -> Tensor:
        return x * torch.sigmoid(self.beta * x)


class ResidualBlock(nn.Module):
    def __init__(self, width: int, use_adaptive_activation: bool) -> None:
        super().__init__()
        self.linear1 = nn.Linear(width, width)
        self.linear2 = nn.Linear(width, width)
        self.act1 = AdaptiveSwish(width) if use_adaptive_activation else nn.Tanh()
        self.act2 = AdaptiveSwish(width) if use_adaptive_activation else nn.Tanh()

    def forward(self, x: Tensor) -> Tensor:
        h = self.act1(self.linear1(x))
        h = self.linear2(h)
        return self.act2(h + x)


class SineLayer(nn.Module):
    """SIREN layer with paper-accurate initialization (Sitzmann et al., 2020)."""

    def __init__(self, in_features: int, out_features: int, omega0: float, is_first: bool) -> None:
        super().__init__()
        self.omega0 = omega0
        self.linear = nn.Linear(in_features, out_features)
        with torch.no_grad():
            bound = (1.0 / in_features) if is_first else (math.sqrt(6.0 / in_features) / omega0)
            self.linear.weight.uniform_(-bound, bound)
            nn.init.zeros_(self.linear.bias)

    def forward(self, x: Tensor) -> Tensor:
        return torch.sin(self.omega0 * self.linear(x))


class PlumeInversionPINN(nn.Module):
    """Concentration field regressor C(x, y, t) with a switchable backbone
    (see ModelConfig.arch_type). All variants keep the same [B,3] -> [B,1]
    interface and non-negativity (softplus) output as the original model, so
    checkpoints, DataParallel wrapping, and the reload test below all still work."""

    def __init__(self, model_config: ModelConfig) -> None:
        super().__init__()
        self.config = model_config
        arch = model_config.arch_type
        in_dim = 3

        if arch == 'fourier_residual':
            self.encoder: nn.Module | None = FourierFeatures(in_dim, model_config.fourier_bands, model_config.fourier_sigma)
            feat_dim = 2 * model_config.fourier_bands
        else:
            self.encoder = None
            feat_dim = in_dim

        if arch == 'siren':
            n_hidden = max(model_config.hidden_layers - 1, 1)
            siren_layers = [SineLayer(in_dim, model_config.hidden_units, model_config.siren_omega0_first, is_first=True)]
            siren_layers += [
                SineLayer(model_config.hidden_units, model_config.hidden_units, model_config.siren_omega0_hidden, is_first=False)
                for _ in range(n_hidden)
            ]
            self.backbone: nn.Module = nn.Sequential(*siren_layers)
            self.blocks = None
            out_width = model_config.hidden_units
        elif arch in ('residual', 'fourier_residual'):
            self.input_proj = nn.Linear(feat_dim, model_config.hidden_units)
            self.input_act = AdaptiveSwish(model_config.hidden_units) if model_config.use_adaptive_activation else nn.Tanh()
            n_blocks = max(model_config.hidden_layers // 2, 1)
            self.blocks = nn.ModuleList(
                [ResidualBlock(model_config.hidden_units, model_config.use_adaptive_activation) for _ in range(n_blocks)]
            )
            out_width = model_config.hidden_units
        else:  # 'mlp' - original baseline architecture, kept selectable for comparison
            layers: list[nn.Module] = []
            width = feat_dim
            for _ in range(model_config.hidden_layers):
                layers.append(nn.Linear(width, model_config.hidden_units))
                layers.append(AdaptiveSwish(model_config.hidden_units) if model_config.use_adaptive_activation else nn.Tanh())
                width = model_config.hidden_units
            self.backbone = nn.Sequential(*layers)
            self.blocks = None
            out_width = model_config.hidden_units

        self.output_layer = nn.Linear(out_width, 1)
        self._initialize_weights()

    def _initialize_weights(self) -> None:
        skip_ids = set()
        if self.config.arch_type == 'siren':
            for module in self.backbone:
                skip_ids.add(id(module.linear))
        for module in self.modules():
            if isinstance(module, nn.Linear) and id(module) not in skip_ids:
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, features: Tensor) -> Tensor:
        if features.ndim != 2 or features.shape[1] != 3:
            raise ValueError('Expected features with shape [batch, 3]')
        arch = self.config.arch_type
        if arch == 'siren':
            hidden = self.backbone(features)
        elif arch in ('residual', 'fourier_residual'):
            encoded = self.encoder(features) if self.encoder is not None else features
            hidden = self.input_act(self.input_proj(encoded))
            for block in self.blocks:
                hidden = block(hidden)
        else:
            encoded = self.encoder(features) if self.encoder is not None else features
            hidden = self.backbone(encoded)
        return F.softplus(self.output_layer(hidden), beta=1.0)


def unwrap_model(model: nn.Module) -> nn.Module:
    return model.module if isinstance(model, nn.DataParallel) else model


# ============================================================
# Physics residual (vectorized, numerically stable float32 autograd)
# ============================================================
def compute_pde_residual(
    features: Tensor, model: nn.Module, u: float, v: float, diffusion: float, reduction: Literal['mean', 'none'] = 'mean'
) -> Tensor:
    features = features.detach().clone().requires_grad_(True)
    concentration = model(features)
    ones = torch.ones_like(concentration)
    gradients = torch.autograd.grad(concentration, features, grad_outputs=ones, create_graph=True, retain_graph=True)[0]
    dC_dx, dC_dy, dC_dt = gradients[:, 0:1], gradients[:, 1:2], gradients[:, 2:3]
    d2C_dx2 = torch.autograd.grad(dC_dx, features, grad_outputs=torch.ones_like(dC_dx), create_graph=True, retain_graph=True)[0][:, 0:1]
    d2C_dy2 = torch.autograd.grad(dC_dy, features, grad_outputs=torch.ones_like(dC_dy), create_graph=True, retain_graph=True)[0][:, 1:2]
    residual = dC_dt + u * dC_dx + v * dC_dy - diffusion * (d2C_dx2 + d2C_dy2)
    squared = residual.square()
    return squared.mean() if reduction == 'mean' else squared


# ============================================================
# Quasi-random (Sobol) collocation / boundary sampling
# Lower-discrepancy coverage of the domain than uniform random sampling for
# the same point budget, which improves PDE-residual estimation stability.
# ============================================================
_collocation_engine = SobolEngine(dimension=3, scramble=True, seed=CONFIG.random_seed)
_boundary_engine = SobolEngine(dimension=2, scramble=True, seed=CONFIG.random_seed + 1)
_rar_pool_engine = SobolEngine(dimension=3, scramble=True, seed=CONFIG.random_seed + 2)


def sample_collocation_points(count: int, device: torch.device) -> Tensor:
    return _collocation_engine.draw(count).to(device=device, dtype=torch.float32)


def sample_boundary_points(count: int, device: torch.device) -> Tensor:
    n = max(count // 4, 1)
    free = _boundary_engine.draw(n * 2).to(device=device, dtype=torch.float32)
    t, other = free[:n, 0:1], free[n:, 0:1]
    side0 = torch.cat([torch.zeros((n, 1), device=device), other, t], dim=1)
    side1 = torch.cat([torch.ones((n, 1), device=device), other, t], dim=1)
    side2 = torch.cat([other, torch.zeros((n, 1), device=device), t], dim=1)
    side3 = torch.cat([other, torch.ones((n, 1), device=device), t], dim=1)
    return torch.cat([side0, side1, side2, side3], dim=0)


def rar_select_points(model: nn.Module, config: TrainingConfig, device: torch.device) -> Tensor:
    """Residual-based Adaptive Refinement: sample a large candidate pool, keep
    the highest-PDE-residual points as new collocation points for the next
    `rar_interval` epochs, concentrating training on the hardest regions."""
    pool_size = config.collocation_points * config.rar_pool_multiplier
    candidates = _rar_pool_engine.draw(pool_size).to(device=device, dtype=torch.float32)
    residuals = compute_pde_residual(candidates, model, config.wind_u, config.wind_v, config.diffusion, reduction='none').detach()
    top_k = max(int(config.collocation_points * config.rar_fraction), 1)
    top_idx = torch.topk(residuals.squeeze(-1), k=min(top_k, pool_size), largest=True).indices
    return candidates[top_idx].detach()


# ============================================================
# SoftAdapt adaptive loss balancing (Heydari et al., 2019)
# Reweights {data, physics, boundary} losses each epoch by their normalized
# rate of change, so whichever term is improving the slowest gets more
# gradient signal - removing the need to hand-tune lambda_* by trial and error.
# ============================================================
class SoftAdaptWeighter:
    def __init__(self, names: list[str], beta: float, floor: float, ceil: float) -> None:
        self.names = names
        self.beta = beta
        self.floor = floor
        self.ceil = ceil
        self.prev: dict[str, float] | None = None

    def compute(self, current: dict[str, float]) -> dict[str, float]:
        if self.prev is None:
            weights = {n: 1.0 for n in self.names}
        else:
            rates = np.array([(current[n] - self.prev[n]) / (abs(self.prev[n]) + 1e-8) for n in self.names])
            shifted = rates - rates.max()
            exp = np.exp(self.beta * shifted)
            raw = np.clip(exp / exp.sum() * len(self.names), self.floor, self.ceil)
            weights = dict(zip(self.names, raw.tolist()))
        self.prev = dict(current)
        return weights


# ============================================================
# Exponential Moving Average of model weights
# Evaluated/checkpointed in place of raw final-step weights: EMA smooths out
# late-training noise and consistently generalizes better for PINNs.
# ============================================================
class EMA:
    def __init__(self, model: nn.Module, decay: float) -> None:
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in unwrap_model(model).state_dict().items()}

    def update(self, model: nn.Module) -> None:
        for k, v in unwrap_model(model).state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model: nn.Module) -> None:
        unwrap_model(model).load_state_dict(self.shadow, strict=True)


def build_warmup_cosine_lambda(warmup_epochs: int, total_epochs: int, min_lr_ratio: float):
    def fn(epoch: int) -> float:
        if epoch < warmup_epochs:
            return (epoch + 1) / max(1, warmup_epochs)
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        cosine = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine

    return fn


base_model = PlumeInversionPINN(MODEL_CONFIG).to(DEVICE)
if GPU_COUNT >= 2:
    model: nn.Module = nn.DataParallel(base_model)
    print('Using torch.nn.DataParallel across', GPU_COUNT, 'GPUs')
else:
    model = base_model
    print('Using single-device model')

sum_params = sum(p.numel() for p in model.parameters())
print(f'Architecture: {MODEL_CONFIG.arch_type} | Trainable parameters: {sum_params}')


In [ ]:
def train_inversion_model(
    model: nn.Module, dataset: SyntheticSensorDataset, config: TrainingConfig, device: torch.device
) -> tuple[list[dict[str, float]], EMA]:
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, build_warmup_cosine_lambda(config.warmup_epochs, config.epochs, config.min_lr_ratio)
    )
    weighter = SoftAdaptWeighter(
        ['data', 'physics', 'boundary'], config.softadapt_beta, config.softadapt_floor, config.softadapt_ceil
    )
    ema = EMA(model, config.ema_decay)

    base_lambdas = {'data': config.lambda_data, 'physics': config.lambda_physics, 'boundary': config.lambda_boundary}
    rar_points: Tensor | None = None
    history: list[dict[str, float]] = []
    start_time = time.time()
    print(
        f'Stage 1/2 (AdamW): {config.epochs} epochs, collocation_points={config.collocation_points}, '
        f'boundary_points={config.boundary_points}, warmup_epochs={config.warmup_epochs}'
    )
    for epoch in range(1, config.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        predicted = model(dataset.features)
        data_loss = F.mse_loss(predicted, dataset.concentration)

        n_random = config.collocation_points if rar_points is None else max(config.collocation_points - rar_points.shape[0], 0)
        random_colloc = sample_collocation_points(n_random, device)
        collocation = random_colloc if rar_points is None else torch.cat([random_colloc, rar_points], dim=0)
        physics_loss = compute_pde_residual(collocation, model, config.wind_u, config.wind_v, config.diffusion)

        boundary = sample_boundary_points(config.boundary_points, device)
        boundary_loss = model(boundary).square().mean()

        current_losses = {
            'data': float(data_loss.detach().cpu()),
            'physics': float(physics_loss.detach().cpu()),
            'boundary': float(boundary_loss.detach().cpu()),
        }
        if epoch > config.softadapt_warmup_epochs:
            adaptive_weights = weighter.compute(current_losses)
        else:
            weighter.prev = dict(current_losses)
            adaptive_weights = {'data': 1.0, 'physics': 1.0, 'boundary': 1.0}

        total_loss = (
            base_lambdas['data'] * adaptive_weights['data'] * data_loss
            + base_lambdas['physics'] * adaptive_weights['physics'] * physics_loss
            + base_lambdas['boundary'] * adaptive_weights['boundary'] * boundary_loss
        )
        total_loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip_norm)
        optimizer.step()
        ema.update(model)
        scheduler.step()

        if epoch > config.rar_warmup_epochs and epoch % config.rar_interval == 0:
            rar_points = rar_select_points(model, config, device)

        record = {
            'epoch': float(epoch),
            'total_loss': float(total_loss.detach().cpu()),
            'data_loss': current_losses['data'],
            'physics_loss': current_losses['physics'],
            'boundary_loss': current_losses['boundary'],
            'weight_data': adaptive_weights['data'],
            'weight_physics': adaptive_weights['physics'],
            'weight_boundary': adaptive_weights['boundary'],
            'learning_rate': float(scheduler.get_last_lr()[0]),
            'grad_norm': float(grad_norm),
        }
        history.append(record)
        if epoch == 1 or epoch % config.log_every == 0 or epoch == config.epochs:
            elapsed = time.time() - start_time
            print(
                f"Epoch {epoch:04d} | total={record['total_loss']:.8f} | data={record['data_loss']:.8f} | "
                f"physics={record['physics_loss']:.8f} | boundary={record['boundary_loss']:.8f} | "
                f"lr={record['learning_rate']:.2e} | grad_norm={record['grad_norm']:.3f} | elapsed={elapsed:.1f}s"
            )
            mlflow_log_metrics(
                {k: v for k, v in record.items() if k != 'epoch'}, step=epoch
            )

    # ------------------------------------------------------------
    # Stage 2/2: full-batch L-BFGS fine-tune for high-precision convergence
    # once AdamW has found a good basin. Uses the final SoftAdapt weights
    # (frozen) since L-BFGS re-evaluates the same closure many times per step.
    # ------------------------------------------------------------
    final_weights = {
        'data': base_lambdas['data'] * history[-1]['weight_data'],
        'physics': base_lambdas['physics'] * history[-1]['weight_physics'],
        'boundary': base_lambdas['boundary'] * history[-1]['weight_boundary'],
    }
    print(f'Stage 2/2 (L-BFGS): {config.lbfgs_steps} steps')
    lbfgs = torch.optim.LBFGS(
        model.parameters(), lr=config.lbfgs_lr, max_iter=config.lbfgs_steps, line_search_fn='strong_wolfe'
    )
    lbfgs_fixed_collocation = sample_collocation_points(config.collocation_points, device)
    lbfgs_fixed_boundary = sample_boundary_points(config.boundary_points, device)

    def closure() -> Tensor:
        lbfgs.zero_grad(set_to_none=True)
        predicted = model(dataset.features)
        data_loss = F.mse_loss(predicted, dataset.concentration)
        physics_loss = compute_pde_residual(lbfgs_fixed_collocation, model, config.wind_u, config.wind_v, config.diffusion)
        boundary_loss = model(lbfgs_fixed_boundary).square().mean()
        loss = final_weights['data'] * data_loss + final_weights['physics'] * physics_loss + final_weights['boundary'] * boundary_loss
        loss.backward()
        return loss

    lbfgs_loss = lbfgs.step(closure)
    ema.update(model)
    lbfgs_total = float(lbfgs_loss.detach().cpu()) if torch.is_tensor(lbfgs_loss) else float(lbfgs_loss)
    print(f'L-BFGS final total_loss={lbfgs_total:.8f}')
    history.append({
        'epoch': float(config.epochs + 1),
        'total_loss': lbfgs_total,
        'data_loss': float('nan'),
        'physics_loss': float('nan'),
        'boundary_loss': float('nan'),
        'weight_data': final_weights['data'],
        'weight_physics': final_weights['physics'],
        'weight_boundary': final_weights['boundary'],
        'learning_rate': 0.0,
        'grad_norm': float('nan'),
    })
    mlflow_log_metrics({'lbfgs_total_loss': lbfgs_total}, step=config.epochs + 1)

    return history, ema


mlflow_start_run(run_name=f'plumetrace_{MODEL_CONFIG.arch_type}_{int(time.time())}')
mlflow_log_params({**asdict(CONFIG), **{f'model_{k}': v for k, v in asdict(MODEL_CONFIG).items()}})

history, ema = train_inversion_model(model, dataset, CONFIG, DEVICE)
history_df = pd.DataFrame(history)
history_df.tail()


In [ ]:
def normalize_scores(scores: np.ndarray) -> np.ndarray:
    clipped = np.clip(scores.astype(np.float64), 0.0, None)
    total = float(clipped.sum())
    if not np.isfinite(total) or total <= 0.0:
        return np.full_like(clipped, 1.0 / clipped.size, dtype=np.float64)
    return clipped / total

def evaluate_pinn_probability_grid(model: nn.Module, sector: CitySector, device: torch.device, grid_size: int) -> dict[str, np.ndarray]:
    model.eval()
    x_axis = np.linspace(0.0, 1.0, grid_size, dtype=np.float32)
    y_axis = np.linspace(0.0, 1.0, grid_size, dtype=np.float32)
    x_grid, y_grid = np.meshgrid(x_axis, y_axis)
    t_grid = np.zeros_like(x_grid, dtype=np.float32)
    features = torch.tensor(np.column_stack([x_grid.ravel(), y_grid.ravel(), t_grid.ravel()]), dtype=torch.float32, device=device)
    preds: list[np.ndarray] = []
    with torch.no_grad():
        for chunk in torch.split(features, 32768):
            preds.append(model(chunk).detach().cpu().numpy())
    field = np.concatenate(preds, axis=0).reshape(grid_size, grid_size)
    probabilities = normalize_scores(field)
    latitudes, longitudes = normalized_to_lat_lon(x_grid, y_grid, sector)
    return {'latitudes': latitudes, 'longitudes': longitudes, 'probabilities': probabilities, 'field': field}

def evaluate_physics_fused_posterior(pinn_map: dict[str, np.ndarray], dataset: SyntheticSensorDataset, config: TrainingConfig, temperature: float = 0.010) -> dict[str, np.ndarray]:
    rows = pd.DataFrame(dataset.rows)
    obs_x = rows['x'].to_numpy(dtype=np.float32)[None, :]
    obs_y = rows['y'].to_numpy(dtype=np.float32)[None, :]
    obs_t = rows['elapsed_time'].to_numpy(dtype=np.float32)[None, :]
    obs_c = rows['normalized_concentration'].to_numpy(dtype=np.float32)[None, :]
    source_x = ((pinn_map['longitudes'] - SECTOR.lon_min) / (SECTOR.lon_max - SECTOR.lon_min)).reshape(-1, 1).astype(np.float32)
    source_y = ((pinn_map['latitudes'] - SECTOR.lat_min) / (SECTOR.lat_max - SECTOR.lat_min)).reshape(-1, 1).astype(np.float32)
    signal = analytic_advection_diffusion_plume(obs_x, obs_y, obs_t, source_x, source_y, config.wind_u, config.wind_v, config.diffusion)
    signal = signal / np.maximum(signal.max(axis=1, keepdims=True), 1.0e-8)
    mse = np.mean((signal - obs_c) ** 2, axis=1).reshape(pinn_map['probabilities'].shape)
    likelihood = np.exp(-mse / max(temperature, 1.0e-6))
    neural_prior = np.power(normalize_scores(pinn_map['probabilities']) + 1.0e-12, 0.35)
    fused = normalize_scores(neural_prior * likelihood)
    return {**pinn_map, 'probabilities': fused, 'candidate_mse': mse, 'physics_likelihood': likelihood}

def estimate_peak(probability_map: dict[str, np.ndarray]) -> dict[str, float]:
    p = probability_map['probabilities']
    idx = np.unravel_index(int(np.argmax(p)), p.shape)
    lat = float(probability_map['latitudes'][idx])
    lon = float(probability_map['longitudes'][idx])
    distance = haversine_m(SECTOR.source_latitude, SECTOR.source_longitude, lat, lon)
    return {'latitude': lat, 'longitude': lon, 'probability': float(p[idx]), 'distance_meters': distance}

# Evaluate with the EMA-smoothed weights rather than the raw final-step
# weights: EMA typically generalizes better and is less sensitive to the last
# few noisy gradient/L-BFGS steps.
ema_model = PlumeInversionPINN(MODEL_CONFIG).to(DEVICE)
ema.copy_to(ema_model)
ema_model.eval()

pinn_map = evaluate_pinn_probability_grid(ema_model, SECTOR, DEVICE, CONFIG.grid_size)
fused_map = evaluate_physics_fused_posterior(pinn_map, dataset, CONFIG)
pinn_peak = estimate_peak(pinn_map)
fused_peak = estimate_peak(fused_map)
print('Raw PINN peak (EMA weights):', pinn_peak)
print('Fused physics+PINN peak:', fused_peak)


In [ ]:
checkpoint_path = OUTPUT_DIR / 'plumetrace_pinn_checkpoint.pt'
probability_map_path = OUTPUT_DIR / 'plumetrace_source_probability_map.npz'
history_path = OUTPUT_DIR / 'plumetrace_training_history.csv'
summary_path = OUTPUT_DIR / 'plumetrace_validation_summary.json'
figure_path = OUTPUT_DIR / 'plumetrace_source_probability.png'
loss_curve_path = OUTPUT_DIR / 'plumetrace_training_curves.png'

# The checkpoint stores the EMA-smoothed weights: this is the model that
# `pinn_map` / `fused_map` / the peak estimates above were computed from.
torch.save(
    {
        'model_state_dict': unwrap_model(ema_model).state_dict(),
        'raw_final_state_dict': unwrap_model(model).state_dict(),
        'model_config': asdict(MODEL_CONFIG),
        'training_config': asdict(CONFIG),
        'city_sector': asdict(SECTOR),
        'sensor_stations': [asdict(s) for s in SENSOR_STATIONS],
        'pinn_peak': pinn_peak,
        'fused_peak': fused_peak,
        'torch_version': torch.__version__,
        'gpu_count': GPU_COUNT,
        'gpu_names': [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)],
    },
    checkpoint_path,
)
np.savez_compressed(
    probability_map_path,
    latitudes=fused_map['latitudes'],
    longitudes=fused_map['longitudes'],
    probabilities=fused_map['probabilities'],
    raw_pinn_probabilities=pinn_map['probabilities'],
    candidate_mse=fused_map['candidate_mse'],
)
history_df.to_csv(history_path, index=False)

# AdamW-stage-only view for the loss-reduction ratio (the appended L-BFGS row
# has NaN component losses by design, since it optimizes a single closure).
adamw_history = [h for h in history if not math.isnan(h['data_loss'])]
validation_summary = {
    'known_source': {'latitude': SECTOR.source_latitude, 'longitude': SECTOR.source_longitude},
    'architecture': MODEL_CONFIG.arch_type,
    'raw_pinn_peak': pinn_peak,
    'fused_physics_pinn_peak': fused_peak,
    'final_losses': history[-1],
    'loss_reduction_ratio': adamw_history[-1]['total_loss'] / adamw_history[0]['total_loss'],
    'lbfgs_final_total_loss': history[-1]['total_loss'],
    'probability_sum': float(fused_map['probabilities'].sum()),
    'all_probabilities_finite': bool(np.isfinite(fused_map['probabilities']).all()),
    'pass_distance_threshold_250m': bool(fused_peak['distance_meters'] <= 250.0),
}
summary_path.write_text(json.dumps(validation_summary, indent=2), encoding='utf-8')

# --- Source probability map (unchanged from the original notebook) ---
plt.figure(figsize=(9, 7))
plt.contourf(fused_map['longitudes'], fused_map['latitudes'], fused_map['probabilities'], levels=32, cmap='inferno')
plt.colorbar(label='Source probability')
plt.scatter([SECTOR.source_longitude], [SECTOR.source_latitude], c='cyan', s=90, marker='*', label='Known source')
plt.scatter([fused_peak['longitude']], [fused_peak['latitude']], c='white', s=70, marker='x', label='Predicted source')
for station in SENSOR_STATIONS:
    plt.scatter([station.longitude], [station.latitude], c='lime', s=35)
    plt.text(station.longitude + 0.00015, station.latitude + 0.00015, station.sensor_id, color='white', fontsize=8)
plt.title(f'PlumeTrace fused physics + PINN source probability ({MODEL_CONFIG.arch_type})')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend(loc='best')
plt.tight_layout()
plt.savefig(figure_path, dpi=180)
plt.show()

# --- New: training diagnostics (loss components, LR, grad norm, adaptive weights) ---
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
epochs_axis = history_df['epoch']

ax = axes[0, 0]
ax.semilogy(epochs_axis, history_df['total_loss'], label='total', linewidth=1.5)
ax.semilogy(epochs_axis, history_df['data_loss'], label='data', alpha=0.8)
ax.semilogy(epochs_axis, history_df['physics_loss'], label='physics', alpha=0.8)
ax.semilogy(epochs_axis, history_df['boundary_loss'], label='boundary', alpha=0.8)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (log scale)'); ax.set_title('Loss components'); ax.legend(fontsize=8)

ax = axes[0, 1]
ax.plot(epochs_axis, history_df['weight_data'], label='w_data', alpha=0.8)
ax.plot(epochs_axis, history_df['weight_physics'], label='w_physics', alpha=0.8)
ax.plot(epochs_axis, history_df['weight_boundary'], label='w_boundary', alpha=0.8)
ax.set_xlabel('Epoch'); ax.set_ylabel('SoftAdapt weight'); ax.set_title('Adaptive loss weights'); ax.legend(fontsize=8)

ax = axes[1, 0]
ax.plot(epochs_axis, history_df['learning_rate'])
ax.set_xlabel('Epoch'); ax.set_ylabel('Learning rate'); ax.set_title('AdamW warmup + cosine schedule')

ax = axes[1, 1]
ax.plot(epochs_axis, history_df['grad_norm'])
ax.set_xlabel('Epoch'); ax.set_ylabel('Gradient norm (pre-clip)'); ax.set_title('Gradient norm')

fig.suptitle(f'PlumeTrace training diagnostics ({MODEL_CONFIG.arch_type})')
fig.tight_layout()
fig.savefig(loss_curve_path, dpi=160)
plt.show()

print(json.dumps(validation_summary, indent=2))
print('Saved checkpoint:', checkpoint_path)
print('Saved probability map:', probability_map_path)
print('Saved training history:', history_path)
print('Saved validation summary:', summary_path)
print('Saved figure:', figure_path)
print('Saved training curves:', loss_curve_path)

for artifact in (checkpoint_path, probability_map_path, history_path, summary_path, figure_path, loss_curve_path):
    mlflow_log_artifact(artifact)
mlflow_end_run()


In [ ]:
# Optional reload test: proves the saved checkpoint is usable in a fresh model instance.
# weights_only=False: this checkpoint bundles config dicts/metadata alongside
# tensors (not just a raw state_dict), and it's a file we just wrote ourselves.
state = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
reloaded_model_config = ModelConfig(**state['model_config'])
reloaded = PlumeInversionPINN(reloaded_model_config).to(DEVICE)
reloaded.load_state_dict(state['model_state_dict'])
reloaded.eval()
test_feature = torch.tensor([[dataset.source_x, dataset.source_y, 0.0]], dtype=torch.float32, device=DEVICE)
with torch.no_grad():
    source_concentration = float(reloaded(test_feature).cpu().item())
print('Reloaded checkpoint OK. Architecture:', reloaded_model_config.arch_type)
print('Predicted normalized C at known source, t=0:', source_concentration)
